<a href="https://colab.research.google.com/github/gitesei/WSL_Simulation_Lab/blob/main/md_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Water Simulation Lab
This Colab notebook enables running molecular dynamics (MD) simulations of pure water and studying structure and dynamics.

MD simulations employ the TIP4P/2005 water model [1].

Simulations are run using OpenMM [2] at the user-defined temperature.

The notebooks guides you through analysis of water density [3], hydrogen bonding, radial distribution function [5], rotational diffusion [6,7] and comparison with corresponding reference experimental data.

### Usage
We recommend running MD simulations run on a single GPU. To enable GPU select `Runtime` from the menu, then `Change runtime type` and select `GPU`.

Note: Cells for preliminary operations should be executed one by one to prevent crashes.

### References

1. J. L. F. Abascal, C. Vega __A general purpose model for the condensed phases of water: TIP4P/2005__ _J. Chem. Phys._ 2005 123(23):234505 DOI: https://doi.org/10.1063/1.2121687

2. P. Eastman, J. Swails, J. D. Chodera et al. __OpenMM 7: Rapid development of high performance algorithms for molecular dynamics__ _PLoS Comput Biol._ 2017 13(7):e1005659 DOI: https://doi.org/10.1371/journal.pcbi.1005659

3. G. S. Kell __Density, Thermal Expansivity, and Compressibility of Liquid Water from 0° to 150°C: Correlations and Tables for Atmospheric Pressure and Saturation Reviewed and Expressed on 1968 Temperature Scale__ _J. Chem. Eng. Data_ 1975 20(1):97–105 DOI: https://doi.org/10.1021/je60064a005

4. A. Luzar, D. Chandler __Effect of Environment on Hydrogen Bond Dynamics in Liquid Water__ _Phys. Rev. Lett._ 1996 76(6):928–931 DOI: https://doi.org/10.1103/PhysRevLett.76.928

5. L. B. Skinner, C. J. Benmore, J. C. Neuefeind, J. B. Parise __The structure of water around the compressibility minimum__ _J. Chem. Phys._ 2014 141(21):214507 DOI: https://doi.org/10.1063/1.4902412

6. D. Laage, G. Stirnemann, F. Sterpone, R. Rey, J. T. Hynes __Reorientation and Allied Dynamics in Water and Aqueous Solutions__ _Annu. Rev. Phys. Chem._ 2011 62:395–416 DOI: https://doi.org/10.1146/annurev.physchem.012809.103503

7. G. Camisasca, N. Galamba, K. T. Wikfeldt, L. G. M. Pettersson __Translational and rotational dynamics of high and low density TIP4P/2005 water__ _J. Chem. Phys._ 2019 150(22):224507 DOI: https://doi.org/10.1063/1.5079956

In [ ]:
# @title 1. Set the environment for simulations and analyses
!pip install openmm[cuda12]
!pip install --upgrade MDAnalysis
!pip install mdtraj
!pip install -q py3Dmol gdown mrcfile &> /dev/null

In [ ]:
# @title 2. Configure your water system

import numpy as np
import pandas as pd
import os
import shutil
import ipywidgets as widgets
from IPython.display import display,Markdown
import warnings
import yaml
import mdtraj as md
import py3Dmol
import io

warnings.filterwarnings('ignore')

#@markdown For direct comparison with the experimental RDFs of Skinner et al. [5], select one of the temperatures available in their supplementary dataset.

temperature = 295.1 #@param [254.1, 263.1, 268.1, 277.1, 284.5, 295.1, 307.0, 312.0, 323.7, 334.1, 342.7, 354.8, 365.9]
box_side_length = 2.20 #@param {type:"number"}
simulation_time = 2 #@param {type:"number"}
#@markdown <i>*Units: temperature [K], box side length [nm], simulation time [ns]<i>

system_name = f"{box_side_length:.2f}_{temperature:.2f}"

if not os.path.isdir(f"{system_name}"):
    os.system(f"mkdir -p {system_name}")
    os.system(f"mkdir -p {system_name}/figures")

N_steps = simulation_time * 1000 / 0.002

config_sim_data = dict(temperature=float(temperature), box_side_length=box_side_length, N_steps=N_steps)
yaml.dump(config_sim_data, open(f'{system_name}/config_sim.yaml','w'))

df_results = pd.DataFrame(index=[system_name])
df_results.index.name = "system"
df_results.loc[system_name,"temperature_sim_K"] = float(config_sim_data["temperature"])

In [ ]:
#@title <b><font color='#A79AB2'>MD simulation toolbox</font></b>

import time
from fastprogress import progress_bar
from openmm import *
from openmm.app import *
from openmm.unit import *

def run_steps(simulation, steps, chunk=1000):
    starttime = time.time()
    nblocks, remainder = divmod(steps, chunk)

    for _ in progress_bar(range(nblocks)):
        simulation.step(chunk)

    if remainder:
        simulation.step(remainder)

    elapsed = time.time()-starttime
    print(f"Simulation time: {elapsed//3600:.0f}h {(elapsed//60)%60:.0f}min {elapsed%60:.2f}s")

def simulate(config):
    temperature = config['temperature']
    box_side_length = config['box_side_length']
    N_steps = int(config['N_steps'])

    top = Topology()
    modeller = Modeller(top, [])

    ff = ForceField('charmm36.xml','charmm36/tip4p2005.xml')
    modeller.addSolvent(ff,model='tip4pew',boxSize=Vec3(box_side_length,box_side_length,box_side_length)*nanometer)

    n_atoms = modeller.topology.getNumAtoms()
    n_residues = modeller.topology.getNumResidues()
    n_waters = sum([res.name in {'HOH','WAT','TIP4'} for res in modeller.topology.residues()])
    n_ions = n_residues-n_waters

    print(f"Number of atoms: {n_atoms}")
    print(f"Number of water molecules: {n_waters}")

    dt = 0.002*picoseconds
    Temperature = temperature*kelvin
    integrator = LangevinMiddleIntegrator(Temperature,1/picosecond,dt)

    system = ff.createSystem(modeller.topology,nonbondedMethod=PME,nonbondedCutoff=1*nanometer,constraints=HBonds)
    simulation = Simulation(modeller.topology,system,integrator)
    simulation.context.setPositions(modeller.positions)

    state = simulation.context.getState(getEnergy=True)
    e_0 = state.getPotentialEnergy()
    print(f"Initial potential energy: {e_0}")

    simulation.minimizeEnergy()

    state = simulation.context.getState(getEnergy=True,getPositions=True)
    e_1 = state.getPotentialEnergy()
    print(f"Potential energy after minimization: {e_1}")
    print(f"Energy change: {e_1-e_0}")

    with open(f'{system_name}/EM_top.pdb','w') as f:
        PDBFile.writeFile(simulation.topology,state.getPositions(),f)

    simulation.context.setVelocitiesToTemperature(Temperature)

    simulation.reporters.append(StateDataReporter(f'{system_name}/NVT_log.txt',1000,step=True,temperature=True,potentialEnergy=True,density=True,volume=True))
    print("Run NVT for 50000 steps")
    run_steps(simulation,50000,1000)
    simulation.reporters.pop()

    system.addForce(MonteCarloBarostat(1*bar,Temperature,25))
    simulation.context.reinitialize(preserveState=True)

    simulation.reporters.append(StateDataReporter(f'{system_name}/NPT_log.txt',500,step=True,temperature=True,potentialEnergy=True,density=True,volume=True))
    simulation.reporters.append(DCDReporter(f'{system_name}/NPT_traj.dcd',500))

    print(f"Run NPT for {N_steps:d} steps")
    run_steps(simulation,N_steps,1000)

In [ ]:
# @title 3. Run MD simulation
config = yaml.safe_load(open(f'{system_name}/config_sim.yaml', 'r'))
simulate(config)

In [ ]:
# @title 4. Visualize the trajectory

#@markdown Here we visualize every 10th frame of the trajectory.

visualization_stride = 10
traj = md.load_dcd(f'{system_name}/NPT_traj.dcd', top=f'{system_name}/EM_top.pdb')[::visualization_stride]

def atom_line(serial, name, resname, resid, x, y, z, element):
    return f"HETATM{serial:5d} {name:<4s} {resname:>3s} A{resid:4d}    {x:8.3f}{y:8.3f}{z:8.3f}  1.00  0.00          {element:>2s}\n"

def conect_line(i, bonded):
    return f"CONECT{i:5d}" + "".join(f"{j:5d}" for j in bonded) + "\n"

def wrap_centered(x, box):
    return x - box*np.floor(x/box + 0.5)

def center_molecules(frame_xyz, top, box_lengths):
    xyz = frame_xyz.copy()

    for residue in top.residues:
        atoms = list(residue.atoms)
        inds = np.array([atom.index for atom in atoms])

        O = None
        for atom in atoms:
            if atom.name == "O":
                O = atom
                break
        if O is None:
            O = atoms[0]

        shift = box_lengths*np.floor(xyz[O.index]/box_lengths + 0.5)
        xyz[inds] -= shift

    return xyz

def frame_to_pdb_string(frame_xyz, top, box_lengths):
    lines = []
    serial = 1

    xyz = center_molecules(frame_xyz, top, box_lengths) * 10.0
    a, b, c = box_lengths * 10.0

    for atom, coord in zip(top.atoms, xyz):
        name = atom.name
        resname = atom.residue.name
        resid = atom.residue.index + 1
        element = atom.element.symbol if atom.element is not None else name[0]
        lines.append(atom_line(serial, name, resname, resid, coord[0], coord[1], coord[2], element))
        serial += 1

    x0, x1 = -a/2, a/2
    y0, y1 = -b/2, b/2
    z0, z1 = -c/2, c/2

    corners = [
        (x0,y0,z0), (x1,y0,z0), (x0,y1,z0), (x0,y0,z1),
        (x1,y1,z0), (x1,y0,z1), (x0,y1,z1), (x1,y1,z1)
    ]

    box_serials = []
    for i, (x, y, z) in enumerate(corners):
        lines.append(atom_line(serial, f"B{i+1}", "BOX", 999, x, y, z, "C"))
        box_serials.append(serial)
        serial += 1

    edges = [
        (0,1), (0,2), (0,3),
        (1,4), (1,5),
        (2,4), (2,6),
        (3,5), (3,6),
        (4,7), (5,7), (6,7)
    ]

    adjacency = {s: [] for s in box_serials}
    for i, j in edges:
        si, sj = box_serials[i], box_serials[j]
        adjacency[si].append(sj)
        adjacency[sj].append(si)

    for s in box_serials:
        lines.append(conect_line(s, adjacency[s]))

    return "".join(lines)

pdb_models = []
for i in range(traj.n_frames):
    pdb_models.append(f"MODEL     {i+1}\n")
    pdb_models.append(frame_to_pdb_string(traj.xyz[i], traj.topology, traj.unitcell_lengths[i]))
    pdb_models.append("ENDMDL\n")

pdb_string = "".join(pdb_models)

view = py3Dmol.view(width=650, height=500)
view.addModelsAsFrames(pdb_string, "pdb", {"keepH": True})
view.setBackgroundColor("white")

view.setStyle({}, {})
view.setStyle({'resn':'HOH', 'atom':'O'},  {'sphere':{'color':'red',   'radius':1.52}})
view.setStyle({'resn':'HOH', 'atom':'H1'}, {'sphere':{'color':'white', 'radius':1.20}})
view.setStyle({'resn':'HOH', 'atom':'H2'}, {'sphere':{'color':'white', 'radius':1.20}})
view.setStyle({'resn':'HOH', 'atom':'M'}, {})
view.setStyle({'resn':'BOX'}, {'stick':{'color':'black', 'radius':0.05}})

view.zoomTo()
view.animate({'loop':'forward', 'reps':1, 'interval':200})
view.show()

In [ ]:
#@title <b><font color='#A79AB2'>Analysis toolbox</font></b>
import matplotlib.pyplot as plt
import MDAnalysis as mda
import MDAnalysis.analysis.rdf as RDF
from MDAnalysis.analysis.hydrogenbonds.hbond_analysis import HydrogenBondAnalysis as HBA

def load_state_data(filename,columns):
    data = np.loadtxt(filename,delimiter=',')
    return {name:data[:,i] for i,name in enumerate(columns)}

def plot_timeseries(x,y,ylabel,title,average=False):
    plt.figure(figsize=(6,4))
    plt.plot(x*1e-3,y)

    if average:
        av = np.mean(y)
        print(f"Average {ylabel}: {av:.3f}")
        plt.axhline(av,color='tab:red',ls='--')

    plt.xlabel("Time (ns)")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(f'{system_name}/figures/{title.replace(' ','_')}.jpg',dpi=600)
    plt.show()

def add_water_bonds(u,oxygen='O',hydrogens=('H1','H2')):
    bonds = []

    for res in u.residues:
        O = res.atoms.select_atoms(f"name {oxygen}")

        for hydrogen in hydrogens:
            H = res.atoms.select_atoms(f"name {hydrogen}")

            if len(O) == 1 and len(H) == 1:
                bonds.append((O[0].index,H[0].index))

    u.add_bonds(bonds)

In [ ]:
# @title 5. Load the trajectory

#@markdown We now load the NPT trajectory and inspect its basic contents.
#@markdown
#@markdown The number of saved trajectory frames is generally much smaller than the number of MD integration steps because coordinates are written only at selected intervals.
#@markdown
#@markdown **Question**
#@markdown
#@markdown 1. Using the total simulation time and the number of saved frames, estimate the time interval in ps between consecutive saved frames.

u = mda.Universe(f"{system_name}/EM_top.pdb",f"{system_name}/NPT_traj.dcd")

print(f"Frames: {len(u.trajectory)}")
print(f"Atoms: {u.atoms.n_atoms}")
print(f"Water molecules: {len(u.select_atoms('resname HOH and name O'))}")
print(f"Simulation time: {simulation_time:g} ns")

In [ ]:
# @title 5. NVT analysis
#@markdown In the NVT ensemble, the number of particles and volume are fixed while the temperature is controlled by a thermostat.
#@markdown
#@markdown **Questions**
#@markdown
#@markdown 1. Does the temperature fluctuate around the target temperature?
#@markdown 2. Is there a systematic increase or decrease in temperature?
#@markdown 3. Why is the density constant during an NVT simulation?

nvt = load_state_data(f"{system_name}/NVT_log.txt",['step','potential_energy','temperature','volume','density'])

plot_timeseries(nvt['step'],nvt['temperature'],"Temperature (K)","NVT temperature")
plot_timeseries(nvt['step'],nvt['density'],"Density (g/mL)","NVT density")

In [ ]:
# @title 6. NPT analysis

#@markdown In the NPT ensemble, pressure and temperature are controlled while the box volume is allowed to fluctuate. Since the number of particles is fixed, fluctuations in volume result in fluctuations in density.
#@markdown
#@markdown We compare the average simulated density with the density of liquid water at the same temperature using the empirical formula from Kell [3],
#@markdown
#@markdown $\rho(T)= \frac{999.83952+16.945176T-7.9870401\times10^{-3}T^2-4.6170461\times10^{-5}T^3+1.0556302\times10^{-7}T^4-2.8054253\times10^{-10}T^5}{1+1.6897850\times10^{-2}T},$
#@markdown
#@markdown where $T$ is the temperature in °C and $\rho$ is obtained in kg m$^{-3}$ = g L$^{-1}$.
#@markdown
#@markdown The first part of the trajectory may correspond to equilibration. Use `discard_time_ps` to exclude an initial time interval from the averages below.
#@markdown
#@markdown The same equilibration time will also be discarded in all subsequent structural and dynamical analyses.
#@markdown
#@markdown **Questions**
#@markdown
#@markdown 1. Does the density fluctuate around a stable average, or does it show a clear trend?
#@markdown 2. Does the volume reach a stationary regime?
#@markdown 3. Are the volume and density fluctuations correlated or anticorrelated?
#@markdown 4. How much of the initial trajectory should be discarded as equilibration?
#@markdown 5. How close is the equilibrated average density to the Kell estimate at the same temperature?
#@markdown 6. Does TIP4P/2005 overestimate or underestimate the experimental density?

discard_time_ps = 100.0 #@param {type:"number"}
#@markdown <i>*Units: equilibration time discarded from this and all subsequent analyses [ps]<i>

npt = load_state_data(f"{system_name}/NPT_log.txt",['step','potential_energy','temperature','volume','density'])

def kell_density(T):
    t = T - 273.15
    return (999.83952 + 16.945176*t - 7.9870401e-3*t**2 - 4.6170461e-5*t**3 + 1.0556302e-7*t**4 - 2.8054253e-10*t**5)/(1 + 1.6897850e-2*t)

dt_md_ps = 0.002
npt_time_ps = (npt["step"]-npt["step"][0])*dt_md_ps
mask_eq = npt_time_ps >= discard_time_ps
discard_frames = (npt_time_ps < discard_time_ps).sum()

T_sim = float(config_sim_data["temperature"])
rho_sim = np.mean(npt["density"][mask_eq])
rho_kell = kell_density(T_sim)/1000
rho_diff = rho_sim - rho_kell
rho_error = 100*rho_diff/rho_kell

plot_timeseries(npt_time_ps,npt['temperature'],"Temperature (K)","NPT temperature",average=True)
plot_timeseries(npt_time_ps,npt['volume'],"Volume (nm$^3$)","NPT volume",average=True)
plot_timeseries(npt_time_ps,npt['density'],"Density (g/mL)","NPT density",average=True)

print(f"Simulation temperature: {T_sim:.1f} K")
print(f"Discarded equilibration time: {discard_time_ps:g} ps")
print(f"Average simulated density: {rho_sim:.4f} g/mL")
print(f"Kell density at {T_sim:g} K: {rho_kell:.4f} g/mL")
print(f"Difference: {rho_diff:+.4f} g/mL ({rho_error:+.2f}%)")

config_sim_data["discard_time_ps"] = float(discard_time_ps)

df_results.loc[system_name,"discard_time_ps"] = discard_time_ps
df_results.loc[system_name,"density_sim_g_mL"] = rho_sim
df_results.loc[system_name,"density_kell_g_mL"] = rho_kell
df_results.loc[system_name,"density_difference_g_mL"] = rho_diff
df_results.loc[system_name,"density_error_percent"] = rho_error

df_results.to_csv(f"{system_name}/analysis_results.csv")

In [ ]:
# @title 6.1 Cumulative average of the density

#@markdown A time series can fluctuate strongly even after the system has reached equilibrium.
#@markdown One useful way to assess whether an average is becoming stable is to calculate the **cumulative average**.
#@markdown
#@markdown At each time $t_n$, the cumulative average includes all density values from the beginning of the trajectory up to that point:
#@markdown
#@markdown $$\overline{\rho}(t_n)=\frac{1}{n}\sum_{i=1}^{n}\rho(t_i).$$
#@markdown
#@markdown Early in the trajectory, the cumulative average can change rapidly because only a small number of frames contribute.
#@markdown As more frames are included, the curve should become progressively less sensitive to individual fluctuations.
#@markdown
#@markdown If the trajectory is sufficiently long and samples a stationary equilibrium distribution, the cumulative average should approach a stable value at long times.
#@markdown A persistent positive or negative slope at long times suggests that the estimated average is still changing and may not yet be converged.
#@markdown
#@markdown Note that a flat cumulative average is a useful convergence check, but it does not by itself prove that all relevant configurations have been sampled.
#@markdown
#@markdown **Questions**
#@markdown
#@markdown 1. How does the cumulative average behave at the beginning of the trajectory?
#@markdown 2. Does it approach a plateau at long times?
#@markdown 3. Is the slope of the cumulative average approximately zero near the end of the trajectory?

density_cumavg = np.cumsum(npt["density"]) / np.arange(1, len(npt_time_ps) + 1)

plt.figure(figsize=(6,3.5))
plt.plot(npt_time_ps/1000, density_cumavg)

plt.xlabel("Time (ns)")
plt.ylabel("Cumulative average density (g/mL)")
plt.title("Cumulative average of NPT density")

plt.show()

In [ ]:
# @title 7. Hydrogen-bond analysis

#@markdown A hydrogen bond is identified using geometric criteria.

#@markdown For two water molecules:

#@markdown - the donor is the oxygen covalently bonded to a hydrogen;
#@markdown - the hydrogen is H1 or H2 on the donor molecule;
#@markdown - the acceptor is the oxygen on another water molecule.

#@markdown A hydrogen bond is counted when:

#@markdown 1. the donor–acceptor distance is smaller than 3.5 Å;
#@markdown 2. the donor–hydrogen–acceptor angle is larger than 140°.

#@markdown The distance criterion ensures that the two water molecules are sufficiently close. The angular criterion selects configurations in which the O–H bond points toward the acceptor oxygen.
#@markdown
#@markdown Here we visualize every 10th frame of the trajectory.
#@markdown
#@markdown **Questions**
#@markdown
#@markdown 1. Why is a distance criterion alone insufficient?
#@markdown 2. What happens to the number of detected hydrogen bonds if the angle cutoff is reduced?
#@markdown 3. What happens if the distance cutoff is increased?
#@markdown 4. Why do hydrogen-bond counts fluctuate with time?

d_a_cutoff = 3.5 #@param {type:"number"}
d_h_a_angle_cutoff = 140 #@param {type:"number"}
#@markdown <i>*Units: donor–acceptor distance cutoff [Å], box side length [nm], donor–hydrogen–acceptor angle cutoff [°]<i>

add_water_bonds(u)
print(f"Bonds added: {len(u.bonds)}")

hbonds = HBA(universe=u,hydrogens_sel='name H1 H2',donors_sel='name O',acceptors_sel='name O',update_selections=False,d_a_cutoff=d_a_cutoff,d_h_a_angle_cutoff=d_h_a_angle_cutoff)
hbonds.run(start=discard_frames,step=10)

counts = hbonds.count_by_time()
n_waters = len(u.select_atoms("resname HOH and name O"))
hbonds_per_water = 2*counts/n_waters

plt.figure(figsize=(6,4))
title = f"Hydrogen bonds ({d_a_cutoff/10:.2f} nm, {d_h_a_angle_cutoff:d} deg)"
plt.plot(np.arange(hbonds_per_water.size)*2e-6*5000,hbonds_per_water)
plt.xlabel("Time (ns)")
plt.ylabel("Hydrogen bonds per water molecule")
plt.title(title)
plt.tight_layout()
plt.savefig(f'{system_name}/figures/{title.replace(' ','_')}.jpg',dpi=600)
plt.show()

print(f"Average hydrogen bonds per water molecule: {hbonds_per_water.mean():.2f}")

# 8. Step-by-step calculation of the radial distribution function

In [ ]:
# @title 8.1: Calculate histogram of oxygen–oxygen distances
#@markdown **Questions**
#@markdown
#@markdown 1. Why do the raw pair counts initially increase with distance?
#@markdown
#@markdown 2. How does changing the histogram bin width affect the smoothness and statistical noise of the distribution?
#@markdown
#@markdown 3. Why do the raw pair counts decrease again near the largest sampled distances?
#@markdown
#@markdown 4. For a cubic box of side length $L$, what is the largest distance for which a complete spherical shell can be sampled using periodic boundary conditions and the minimum-image convention?

bin_width = 0.02 #@param {type:"number"}
#@markdown <i>*Units: bin width [Å]<i>

from MDAnalysis.lib.distances import self_distance_array

oxygen = u.select_atoms("resname HOH and name O")
distances = []

for ts in u.trajectory[discard_frames::10]:
    d = self_distance_array(oxygen.positions,box=ts.dimensions)
    distances.append(d)

distances = np.concatenate(distances)

print(f"Number of distances: {len(distances)}")
print(f"Minimum distance: {distances.min():.2f} Å")

edges = np.arange(0,d.max()+bin_width,bin_width)
counts,edges = np.histogram(distances,bins=edges)
r = 0.5*(edges[:-1]+edges[1:])

plt.figure(figsize=(6,4))
title = "Histogram of O–O distances"
plt.bar(r,counts,width=bin_width)
plt.xlabel(r"$r$ (Å)")
plt.ylabel("Pair counts")
plt.title(title)
plt.tight_layout()
plt.savefig(f'{system_name}/figures/{title.replace(' ','_')}.jpg',dpi=600)
plt.show()

In [ ]:
# @title 8.2: Calculate shell volumes

#@markdown The RDF counts neighbors inside spherical shells between $r_i$ and $r_{i+1}$.
#@markdown
#@markdown The exact volume of each shell is
#@markdown
#@markdown $$\Delta V_i=\frac{4\pi}{3}\left(r_{i+1}^3-r_i^3\right).$$
#@markdown
#@markdown Shells at larger distances have larger volumes, so they contain more particles even in a system of noninteracting particles with uniform bulk density. This geometric effect must be removed when normalizing the raw distance histogram.
#@markdown
#@markdown **Questions**
#@markdown
#@markdown 1. Why does the shell volume increase with distance even though the bin width is constant?
#@markdown
#@markdown 2. For narrow bins, the shell volume can be approximated as $\Delta V \approx 4\pi r^2\Delta r$. How does this explain the initial increase in the raw pair-count histogram?

shell_volumes = 4*np.pi/3*(edges[1:]**3-edges[:-1]**3)

plt.figure(figsize=(6,4))
title = "Volume of spherical shells"
plt.plot(r,shell_volumes)
plt.xlabel(r"$r$ (Å)")
plt.ylabel(r"Shell volume (Å$^3$)")
plt.title(title)
plt.tight_layout()
plt.savefig(f'{system_name}/figures/{title.replace(' ','_')}.jpg',dpi=600)
plt.show()

In [ ]:
# @title 8.3: Normalize the histogram

#@markdown In an NPT simulation, the box volume changes from frame to frame. We therefore normalize the distance histogram separately for every frame.
#@markdown
#@markdown For each frame:
#@markdown
#@markdown 1. We count how many unique O–O pairs fall inside each spherical shell.
#@markdown 2. We divide these counts by the shell volume, $\Delta V(r)$, to obtain the pair density at distance $r$.
#@markdown 3. We divide by the bulk pair density in the same frame:
#@markdown
#@markdown $$\rho_{\mathrm{bulk},t}=\frac{N_{\mathrm{pairs}}}{V_t},$$
#@markdown
#@markdown where $N_{\mathrm{pairs}}=N(N-1)/2$ and $V_t$ is the instantaneous box volume.
#@markdown
#@markdown The RDF for frame $t$ is therefore
#@markdown
#@markdown $$g_t(r)=\frac{N_t(r)/\Delta V(r)}{N_{\mathrm{pairs}}/V_t}.$$
#@markdown
#@markdown Finally, we average the framewise RDFs:
#@markdown
#@markdown $$g(r)=\left\langle g_t(r)\right\rangle_t.$$
#@markdown
#@markdown A value of $g(r)=1$ means that O–O pairs occur at distance $r$ with the same probability as in a system of noninteracting particles with the same bulk density.
#@markdown
#@markdown **Questions**
#@markdown
#@markdown 1. What does $g(r)>1$ tell us about the probability of finding two oxygen atoms separated by distance $r$?
#@markdown
#@markdown 2. Why is $g(r)$ close to zero at very short distances?
#@markdown
#@markdown 3. What structural feature of liquid water is represented by the first maximum of $g_{\mathrm{OO}}(r)$?
#@markdown
#@markdown 4. For a cubic box with side length $L$, what is the largest distance from a central particle for which a complete spherical shell can be sampled using the minimum-image convention?

n_pairs = len(oxygen)*(len(oxygen)-1)/2
g_frames = []

for ts in u.trajectory[discard_frames::10]:
    distances = self_distance_array(oxygen.positions,box=ts.dimensions)
    frame_counts = np.histogram(distances,bins=edges)[0]

    volume = np.prod(ts.dimensions[:3])
    pair_density = frame_counts/shell_volumes
    bulk_pair_density = n_pairs/volume

    g_frames.append(pair_density/bulk_pair_density)

g_frames = np.asarray(g_frames)
g_r = g_frames.mean(axis=0)

plt.figure(figsize=(6,4))
title = "Oxygen–oxygen RDF"
plt.plot(r,g_r)
plt.xlabel(r"$r$ (Å)")
plt.ylabel(r"$g_{\mathrm{OO}}(r)$")
plt.title(title)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{system_name}/figures/{title.replace(' ','_')}.jpg',dpi=600)
plt.show()

In [ ]:
# @title 8.3: Upload experimental RDF data

#@markdown Experimental $g_{\mathrm{OO}}(r)$ data are taken from the supplementary material of Skinner et al. [5].
#@markdown
#@markdown 1. Go to:
#@markdown https://pubs.aip.org/jcp/article-supplement/351929/zip/214507_1_supplements/
#@markdown
#@markdown 2. Download and unzip the supplementary material.
#@markdown
#@markdown 3. Upload the file `measured water g_OO(r) vs T - Skinner Benmore 2014.txt` when prompted below.

from google.colab import files

filename = "measured water g_OO(r) vs T - Skinner Benmore 2014.txt"

if not os.path.exists(filename):
    print(f"Please upload: {filename}")
    files.upload()
else:
    print(f"Found: {filename}")

In [ ]:
# @title 8.5: Comparison with experiment

#@markdown We compare the simulated oxygen–oxygen RDF with experimental data from Skinner et al. [5].
#@markdown
#@markdown The experimental $g_{\mathrm{OO}}(r)$ curves were obtained from X-ray scattering at several temperatures.
#@markdown
#@markdown **Questions**
#@markdown
#@markdown 1. Does the simulation reproduce the position and height of the first peak in $g_{\mathrm{OO}}(r)$?
#@markdown 2. Does it reproduce the position and height of the second peak?
#@markdown 3. Which of these features is reproduced better by the water model?
#@markdown 4. What do the first and second peaks tell us about the local structure of liquid water?

with open(filename,"r") as f:
    text = f.read()

lines = text.splitlines()
header_index = next(i for i,line in enumerate(lines) if line.startswith("T(K)"))
header = lines[header_index].split("\t")
temperatures = [float(x) for x in header[1:] if x.strip() and x.strip().lower() != "err" and not x.lower().startswith("dr")]

df_exp = pd.read_csv(io.StringIO("\n".join(lines[header_index+2:])),sep="\t",header=None)
df_exp = df_exp.dropna(axis=1,how="all")

T_sim = float(config_sim_data["temperature"])
T_exp = min(temperatures,key=lambda T: abs(T-T_sim))
i = temperatures.index(T_exp)

r_exp = pd.to_numeric(df_exp.iloc[:,0],errors="coerce")
g_exp = pd.to_numeric(df_exp.iloc[:,1+2*i],errors="coerce")
valid = r_exp.notna() & g_exp.notna()

plt.figure(figsize=(6,4))
plt.plot(r,g_r,label=f"TIP4P/2005 simulation ({T_sim:g} K)")
plt.plot(r_exp[valid],g_exp[valid],label=f"Experiment ({T_exp:g} K)")
plt.xlabel(r"$r$ (Å)")
plt.ylabel(r"$g_{\mathrm{OO}}(r)$")
title = "Oxygen–oxygen RDF: simulation vs experiment"
plt.title(title)
plt.grid(alpha=0.3)
plt.legend()
plt.xlim(0,10)
plt.tight_layout()
plt.savefig(f"{system_name}/figures/{title.replace(' ','_')}.jpg",dpi=600)
plt.show()

print(f"Simulation temperature: {T_sim:g} K")
print(f"Closest experimental temperature: {T_exp:g} K")

df_results.loc[system_name,"temperature_exp_RDF_K"] = T_exp

In [ ]:
df_results.columns

In [ ]:
# @title 8.6: Integrate the first and second O-O neighbor regions

#@markdown Skinner et al. analyze the O–O structure by integrating the excess coordination relative to the bulk:
#@markdown
#@markdown $$\Delta N(r_1,r_2)=4\pi\rho_{\mathrm O}\int_{r_1}^{r_2}r^2[g_{\mathrm{OO}}(r)-1]\,dr.$$
#@markdown
#@markdown Here $\rho_{\mathrm O}$ is the oxygen number density, calculated from the mass density as
#@markdown
#@markdown $$\rho_{\mathrm O}=\frac{\rho_{\mathrm{mass}}}{M_{\mathrm{H_2O}}}\frac{N_A}{10^{30}},$$
#@markdown
#@markdown where $\rho_{\mathrm{mass}}$ is in kg m$^{-3}$, $M_{\mathrm{H_2O}}$ is in kg mol$^{-1}$, and $\rho_{\mathrm O}$ is obtained in Å$^{-3}$.
#@markdown
#@markdown For the simulation, we use the average density in g/L from our NPT simulation.
#@markdown
#@markdown For the experiment, we calculate the liquid-water density at the selected experimental temperature using the empirical correlation from Kell [3].
#@markdown
#@markdown Skinner et al. use 2.4–3.3 Å and 3.3–5.6 Å for the first- and second-neighbor O–O regions. The shaded regions in the plot show the intervals used for the integrations.
#@markdown
#@markdown **Questions**
#@markdown
#@markdown 1. How does the excess coordination in the first-neighbor region compare between simulation and experiment?
#@markdown 2. How does the excess coordination in the second-neighbor region compare?
#@markdown 3. Which region is reproduced better by TIP4P/2005?
#@markdown 4. How sensitive are the results to the integration limits?

first_shell_min = 2.4 #@param {type:"number"}
first_shell_max = 3.3 #@param {type:"number"}
second_shell_min = 3.3 #@param {type:"number"}
second_shell_max = 5.6 #@param {type:"number"}
#@markdown <i>*Units: integration limits [Å]<i>

from scipy.integrate import trapezoid
from scipy.constants import Avogadro

M_water = 0.01801528

def kell_density(T):
    t = T - 273.15
    return (999.83952 + 16.945176*t - 7.9870401e-3*t**2 - 4.6170461e-5*t**3 + 1.0556302e-7*t**4 - 2.8054253e-10*t**5)/(1 + 1.6897850e-2*t)

def oxygen_number_density(rho_mass):
    return rho_mass/M_water*Avogadro/1e30

def excess_coordination(r_values,g_values,rmin,rmax,rho):
    mask = (r_values >= rmin) & (r_values <= rmax)
    return 4*np.pi*rho*trapezoid(r_values[mask]**2*(g_values[mask]-1),r_values[mask])

rho_sim_mass = df_results.loc[system_name].density_sim_g_mL*1000
rho_sim = oxygen_number_density(rho_sim_mass)

rho_exp_mass = kell_density(T_exp)
rho_exp = oxygen_number_density(rho_exp_mass)

r_exp_np = r_exp[valid].to_numpy()
g_exp_np = g_exp[valid].to_numpy()

dN1_sim = excess_coordination(r,g_r,first_shell_min,first_shell_max,rho_sim)
dN2_sim = excess_coordination(r,g_r,second_shell_min,second_shell_max,rho_sim)
dN1_exp = excess_coordination(r_exp_np,g_exp_np,first_shell_min,first_shell_max,rho_exp)
dN2_exp = excess_coordination(r_exp_np,g_exp_np,second_shell_min,second_shell_max,rho_exp)

plt.figure(figsize=(6,4))
plt.plot(r,g_r,label=f"TIP4P/2005 simulation ({T_sim:g} K)")
plt.plot(r_exp_np,g_exp_np,label=f"Experiment ({T_exp:g} K)")
plt.axvspan(first_shell_min,first_shell_max,alpha=0.15,
            label="First-neighbor region",color='tab:blue',lw=0)
plt.axvspan(second_shell_min,second_shell_max,alpha=0.15,
            label="Second-neighbor region",color='tab:green',lw=0)
plt.xlabel(r"$r$ (Å)")
plt.ylabel(r"$g_{\mathrm{OO}}(r)$")
title = "Oxygen–oxygen RDF and integration regions"
plt.title(title)
plt.grid(alpha=0.3)
plt.legend()
plt.xlim(0,10)
plt.tight_layout()
plt.savefig(f'{system_name}/figures/{title.replace(' ','_')}.jpg',dpi=600)
plt.show()

results = f"""O-O excess coordination relative to bulk

Simulation temperature: {T_sim:g} K
Experimental temperature: {T_exp:g} K

Mass densities:
Simulation: {rho_sim_mass:.2f} kg/m^3
Experiment (Kell): {rho_exp_mass:.2f} kg/m^3

Integration intervals:
First neighbor: {first_shell_min:.2f}-{first_shell_max:.2f} Å
Second neighbor: {second_shell_min:.2f}-{second_shell_max:.2f} Å

Simulation:
First neighbor: {dN1_sim:.3f}
Second neighbor: {dN2_sim:.3f}

Experiment:
First neighbor: {dN1_exp:.3f}
Second neighbor: {dN2_exp:.3f}
"""

print(results)

df_results.loc[system_name,"temperature_exp_RDF_K"] = T_exp
df_results.loc[system_name,"first_shell_min_A"] = first_shell_min
df_results.loc[system_name,"first_shell_max_A"] = first_shell_max
df_results.loc[system_name,"second_shell_min_A"] = second_shell_min
df_results.loc[system_name,"second_shell_max_A"] = second_shell_max

df_results.loc[system_name,"excess_coord_first_sim"] = dN1_sim
df_results.loc[system_name,"excess_coord_second_sim"] = dN2_sim
df_results.loc[system_name,"excess_coord_first_exp"] = dN1_exp
df_results.loc[system_name,"excess_coord_second_exp"] = dN2_exp
df_results.to_csv(f"{system_name}/analysis_results.csv")

In [ ]:
# @title 9. Rotational dynamics

#@markdown Molecular reorientation can be quantified using an orientational time-correlation function (OTCF). Following Camisasca et al. [7], for a body-fixed unit vector $\mathbf{u}(t)$ we define
#@markdown
#@markdown $$C_l(t)=\left\langle P_l\left[\mathbf{u}(t_0)\cdot\mathbf{u}(t_0+t)\right]\right\rangle_{t_0,\mathrm{molecules}},$$
#@markdown
#@markdown where $P_l$ is the Legendre polynomial of order $l$. For the first two orders,
#@markdown
#@markdown $$P_1(x)=x,\qquad C_1(t)=\langle\cos\theta(t)\rangle,$$
#@markdown
#@markdown $$P_2(x)=\frac{1}{2}(3x^2-1),\qquad C_2(t)=\left\langle\frac{1}{2}\left[3\cos^2\theta(t)-1\right]\right\rangle.$$
#@markdown
#@markdown $C_1(t)$ and $C_2(t)$ measure orientational memory in different ways and therefore generally decay on different timescales. Higher-order Legendre polynomials are more sensitive to angular changes and their orientational correlation functions generally decay faster.
#@markdown
#@markdown A characteristic measure of the reorientation time is the integral orientational correlation time
#@markdown
#@markdown $$\tau_l=\int_0^\infty C_l(t)\,dt.$$
#@markdown
#@markdown In a finite trajectory, we approximate this as
#@markdown
#@markdown $$\tau_l(t_{\max})=\int_0^{t_{\max}} C_l(t)\,dt,$$
#@markdown
#@markdown where $t_{\max}$ is set by `max_lag_ps`. The value should be sufficiently large for $C_l(t)$ to have decayed close to zero. At lower temperatures, slower reorientation may require a longer integration window.
#@markdown
#@markdown The integral starts at $t=0$ and therefore includes the initial fast librational contribution as well as the slower molecular reorientation. Camisasca et al. distinguish these integral correlation times from relaxation times obtained by fitting the longer-time decay after the initial librational regime.
#@markdown
#@markdown In the Debye rotational-diffusion model, molecular orientation undergoes isotropic rotational Brownian motion through many small angular steps. The model predicts
#@markdown
#@markdown $$C_l(t)=e^{-l(l+1)D_rt},$$
#@markdown
#@markdown and therefore
#@markdown
#@markdown $$\tau_l=\frac{1}{l(l+1)D_r}.$$
#@markdown
#@markdown For the first two orders,
#@markdown
#@markdown $$\tau_1=\frac{1}{2D_r},\qquad \tau_2=\frac{1}{6D_r},$$
#@markdown
#@markdown so that purely Debye rotational diffusion predicts
#@markdown
#@markdown $$\frac{\tau_1}{\tau_2}=3.$$

max_lag_ps = 40.0 #@param {type:"number"}
analysis_stride = 1 #@param {type:"integer"}
#@markdown <i>*Units: maximum correlation time [ps]<i>

from MDAnalysis.lib.distances import minimize_vectors
from scipy.integrate import trapezoid

def P1(x):
    return x

def P2(x):
    return 0.5*(3*x**2-1)

def orientational_correlation(vectors,legendre,max_lag):
    C = np.empty(max_lag+1)
    for lag in range(max_lag+1):
        dots = np.sum(vectors[:len(vectors)-lag]*vectors[lag:],axis=2)
        C[lag] = np.mean(legendre(dots))
    return C

def normalize_vectors(v):
    return v/np.linalg.norm(v,axis=1)[:,None]

In [ ]:
# @title 9.1. Rotational dynamics of the O-H bond

#@markdown Following Camisasca et al. [7], we use the O–H bond as the molecular orientation vector. Both O–H bonds of every water molecule are included in the average.
#@markdown
#@markdown We calculate $C_1^{OH}(t)$ and $C_2^{OH}(t)$ and integrate them to obtain $\tau_1^{OH}$ and $\tau_2^{OH}$. We then use the Debye expressions to calculate the rotational diffusion coefficient that would correspond to each integral correlation time.
#@markdown
#@markdown When interpreting the curves, notice that water reorientation is not described by a single simple Debye process. The short-time decay contains fast librational motion, while longer-time orientational relaxation involves rearrangement of the hydrogen-bond network. Camisasca et al. show that the longer-time $C_2(t)$ of average TIP4P/2005 water is better described by more than one relaxation component.
#@markdown
#@markdown The Debye prediction $\tau_1/\tau_2=3$ therefore provides a useful reference rather than an expected result.
#@markdown
#@markdown **Questions**
#@markdown
#@markdown 1. Which correlation function, $C_1^{OH}(t)$ or $C_2^{OH}(t)$, decays faster?
#@markdown 3. Have both correlation functions decayed close to zero by `max_lag_ps`?
#@markdown 4. What are the integral orientational correlation times $\tau_1^{OH}$ and $\tau_2^{OH}$?
#@markdown 5. How does $\tau_1^{OH}/\tau_2^{OH}$ compare with the Debye prediction of 3?
#@markdown 6. Do the rotational diffusion coefficients inferred independently from $\tau_1^{OH}$ and $\tau_2^{OH}$ agree?

O = u.select_atoms("name O")
H1 = u.select_atoms("name H1")
H2 = u.select_atoms("name H2")

oh_vectors = []

for ts in u.trajectory[discard_frames::analysis_stride]:
    v1 = minimize_vectors(H1.positions-O.positions,ts.dimensions)
    v2 = minimize_vectors(H2.positions-O.positions,ts.dimensions)
    v1 = normalize_vectors(v1)
    v2 = normalize_vectors(v2)
    oh_vectors.append(np.concatenate((v1,v2),axis=0).astype(np.float32))

oh_vectors = np.asarray(oh_vectors)

dt_rot = u.trajectory.dt*analysis_stride
max_lag = min(int(max_lag_ps/dt_rot),len(oh_vectors)-1)
time_rot = np.arange(max_lag+1)*dt_rot

C1_OH = orientational_correlation(oh_vectors,P1,max_lag)
C2_OH = orientational_correlation(oh_vectors,P2,max_lag)

tau1_OH = trapezoid(C1_OH,time_rot)
tau2_OH = trapezoid(C2_OH,time_rot)

Drot_C1_OH = 1/(2*tau1_OH)
Drot_C2_OH = 1/(6*tau2_OH)
tau_ratio_OH = tau1_OH/tau2_OH

plt.figure(figsize=(6,4))
plt.plot(time_rot,C1_OH,label=r"$C_1^{OH}(t)$")
plt.plot(time_rot,C2_OH,label=r"$C_2^{OH}(t)$")
plt.axhline(0,lw=0.8)
plt.xlabel("Time (ps)")
plt.ylabel("Orientational correlation")
title = "O–H orientational correlation functions"
plt.title(title)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(f"{system_name}/figures/{title.replace(' ','_')}.jpg",dpi=600)
plt.show()

print(f"Integral C1 correlation time: {tau1_OH:.3f} ps")
print(f"Integral C2 correlation time: {tau2_OH:.3f} ps")
print(f"tau1/tau2: {tau_ratio_OH:.3f}  (Debye prediction = 3)")
print(f"D_r from C1: {Drot_C1_OH:.4f} ps^-1")
print(f"D_r from C2: {Drot_C2_OH:.4f} ps^-1")

df_results.loc[system_name,"tau1_OH_ps"] = tau1_OH
df_results.loc[system_name,"tau2_OH_ps"] = tau2_OH
df_results.loc[system_name,"tau1_tau2_OH_ratio"] = tau_ratio_OH
df_results.loc[system_name,"Drot_from_C1_OH_ps-1"] = Drot_C1_OH
df_results.loc[system_name,"Drot_from_C2_OH_ps-1"] = Drot_C2_OH
df_results.to_csv(f"{system_name}/analysis_results.csv")

In [ ]:
# @title 9.2. Rotational dynamics of the dipole moment

#@markdown The orientational correlation function also depends on which molecular axis is used to define orientation.
#@markdown
#@markdown In TIP4P/2005, the massless M site lies along the bisector of the H–O–H angle. The O–M vector therefore defines the molecular dipole axis.
#@markdown
#@markdown We calculate $C_2^{OM}(t)$ using exactly the same procedure as for the O–H bond and compare it with $C_2^{OH}(t)$.
#@markdown
#@markdown The corresponding integral correlation times are
#@markdown
#@markdown $$\tau_2^{OH}=\int_0^\infty C_2^{OH}(t)\,dt,$$
#@markdown
#@markdown $$\tau_2^{OM}=\int_0^\infty C_2^{OM}(t)\,dt.$$
#@markdown
#@markdown Since water rotation is anisotropic, different body-fixed molecular axes can have different orientational correlation functions even though they belong to the same rigid molecule.
#@markdown
#@markdown **Questions**
#@markdown
#@markdown 1. Which molecular vector loses orientational memory faster?
#@markdown 2. How do $\tau_2^{OH}$ and $\tau_2^{OM}$ compare?
#@markdown 3. Why have the two vectors different rotational correlation times?

M = u.select_atoms("name M")

om_vectors = []

for ts in u.trajectory[discard_frames::analysis_stride]:
    vm = minimize_vectors(M.positions-O.positions,ts.dimensions)
    vm = normalize_vectors(vm)
    om_vectors.append(vm.astype(np.float32))

om_vectors = np.asarray(om_vectors)

C2_OM = orientational_correlation(om_vectors,P2,max_lag)
tau2_OM = trapezoid(C2_OM,time_rot)

plt.figure(figsize=(6,4))
plt.plot(time_rot,C2_OH,label=r"$C_2^{OH}(t)$")
plt.plot(time_rot,C2_OM,label=r"$C_2^{OM}(t)$")
plt.axhline(0,lw=0.8)
plt.xlabel("Time (ps)")
plt.ylabel(r"$C_2(t)$")
title = "O–H and dipole ACF"
plt.title(title)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(f'{system_name}/figures/{title.replace(' ','_')}.jpg',dpi=600)
plt.show()

print(f"O-H integral C2 correlation time: {tau2_OH:.3f} ps")
print(f"O-M integral C2 correlation time: {tau2_OM:.3f} ps")
print(f"tau2(OH)/tau2(OM): {tau2_OH/tau2_OM:.3f}")

df_results.loc[system_name,"tau2_OM_ps"] = tau2_OM
df_results.loc[system_name,"tau2_OH_tau2_OM_ratio"] = tau2_OH/tau2_OM
df_results.to_csv(f"{system_name}/analysis_results.csv")

In [ ]:
# @title 10. Download results

# @markdown In this zip file:

# @markdown `NPT_traj.dcd`: the NPT trajectory file;

# @markdown `EM_top.pdb`: the energy-minimized topology file;

# @markdown `analysis_results.csv`: summary of simulation results;

# @markdown Plots generated during the analysis.

files_to_download = [f"{system_name}/NPT_traj.dcd",
                     f"{system_name}/EM_top.pdb",
                     f"{system_name}/analysis_results.csv",
                     f"{system_name}/figures"]

import subprocess, glob
from google.colab import files

for filename in glob.glob(f'{system_name}/*'):
    if filename not in files_to_download:
        try:
            os.remove(f'{filename:s}')
        except:
            shutil.rmtree(f'{filename:s}')

zipper = f'zip -r {system_name}.zip {system_name}'
subprocess.run(zipper.split())
files.download(f'{system_name}.zip')